# What this simulation is doing

This notebook runs **2D Lattice Boltzmann (D2Q9)** channel flow (`LatticeBoltzmann`).

- **Shown variables**: `vorticity`, `velocity_x`, `velocity_y`, `rho`.
- **Dynamics**: collision + streaming updates with inflow/outflow and wall effects.

## Initial Conditions

- Fluid starts close to rest (`u≈0, v≈0`) with nearly uniform density.
- Dynamics develop from boundary driving (inflow profile and optional oscillatory forcing), not from a large interior pulse.

## Boundary Conditions

- **Top/Bottom**: no-slip wall behavior (bounce-back).
- **Left boundary**: prescribed inflow (parabolic profile, optionally oscillatory in time).
- **Right boundary**: open/outflow condition (low-reflection approximation).
- **Cylinder obstacle**: optional internal no-slip bounce-back region when enabled.

## What to expect

- With cylinder: wake formation and vortex shedding.
- Without cylinder + oscillatory inlet: unsteady shear-driven structures in the channel.

## Quick parameter guide

- `width,height`: `192x64` (fast) or `256x64` (cleaner wake)
- `T`: choose with `dt` to control internal steps
- `dt`: smaller values reduce frame-to-frame jumpiness
- `n_saved_frames`: set exact returned trajectory length
- `viscosity`: `0.01`–`0.03` (lower → higher Reynolds number)
- `u_in`: `0.06`–`0.12` (keep below `~0.15` for stable LBM)
- `oscillation_frequency`: larger values produce faster inlet oscillations
- `use_cylinder`: `True` for wake shedding, `False` for pure channel case

## Governing equations (D2Q9 Lattice Boltzmann)

The solver advances particle populations $f_i(\mathbf{x},t)$ (for $i=0,\dots,8$) using a BGK collision model:
$$
f_i(\mathbf{x}+\mathbf{c}_i\Delta t,\ t+\Delta t)
=
f_i(\mathbf{x},t)-\omega\left(f_i(\mathbf{x},t)-f_i^{\mathrm{eq}}(\mathbf{x},t)\right),
$$
where $\omega=1/\tau$ and
$$
\nu = \frac{\tau-1/2}{3}
$$
in lattice units.

Macroscopic fields are recovered from moments:
$$
\rho=\sum_i f_i,
\qquad
\rho\mathbf{u}=\sum_i \mathbf{c}_i f_i.
$$

The equilibrium distribution is the standard D2Q9 second-order form
$$
f_i^{\mathrm{eq}} = w_i\rho\left[1+3(\mathbf{c}_i\cdot\mathbf{u})+\tfrac{9}{2}(\mathbf{c}_i\cdot\mathbf{u})^2-\tfrac{3}{2}|\mathbf{u}|^2\right].
$$

### Variable definitions

- $\mathbf{x}=(x,y)$: lattice location.
- $t$: discrete time.
- $i\in\{0,\dots,8\}$: D2Q9 direction index.
- $f_i$: particle distribution in direction $i$.
- $\mathbf{c}_i$: discrete lattice velocity for direction $i$.
- $f_i^{\mathrm{eq}}$: local equilibrium distribution.
- $w_i$: D2Q9 quadrature weights.
- $\rho$: fluid density from zeroth moment of $f_i$.
- $\mathbf{u}=(u,v)$: macroscopic velocity from first moment of $f_i$.
- $\tau$: relaxation time.
- $\omega$: collision frequency ($1/\tau$).
- $\nu$: kinematic viscosity in lattice units.

In the low-Mach regime, this recovers incompressible Navier-Stokes dynamics.

In [ ]:
from autosim.experimental.simulations import LatticeBoltzmann as Sim

# Flow past a cylinder with parabolic inflow (LBM)
sim = Sim(
    return_timeseries=True,
    width=64,
    height=64,
    T=12.8,
    dt=12.8 / 400, # Buffer
    n_saved_frames=321,
    parameters_range={
        "viscosity": (0.01, 0.03),
        "u_in": (0.05, 0.15),
        "oscillation_frequency": (0.1, 2.0),
    },
    log_level="progress_bar",
    use_cylinder=False,
    oscillatory_inlet=True,
    skip_nt=0
)

data = sim.forward_samples_spatiotemporal(n=10, random_seed=None)

In [ ]:
data["data"].shape, data["constant_scalars"]

In [ ]:
from IPython.display import HTML

from autosim.utils import plot_spatiotemporal_video

# LBM channels: vorticity, velocity_x, velocity_y, rho
# Example params: [0.0146, 0.1334, 0.2029]
anim = plot_spatiotemporal_video(
    # data["data"][:, :100],
    data["data"][:, :],
    batch_idx=5,
    channel_names=["vorticity", "velocity_x", "velocity_y", "rho"],
)

HTML(anim.to_jshtml())

In [ ]:
from autosim.utils import generate_output_data

outputs = generate_output_data(sim)

In [ ]:
import os

import torch

for split in ["train", "valid", "test"]:
    os.makedirs(f"../../../autocast/datasets/lattice_boltzmann/{split}/", exist_ok=True)
    torch.save(outputs[split], f"../../../autocast/datasets/lattice_boltzmann/{split}/data.pt")

In [ ]:

# Check shape
data["data"].shape

In [ ]:
from autosim.utils import plot_spatiotemporal_video

# LBM channels: vorticity, velocity_x, velocity_y, rho
anim = plot_spatiotemporal_video(
    outputs["train"]["data"],
    batch_idx=0,
    channel_names=["vorticity", "velocity_x", "velocity_y", "rho"],
)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())